# PDF OCR with Gemini API - Test & Debug Notebook

This notebook tests PDF to text extraction using Google Gemini 2.5 Flash with proper response handling based on latest LangChain & Google API documentation.

**Key Features:**
- ✅ Proper response.content handling (can be string or list)
- ✅ Base64 image encoding for multimodal input
- ✅ Document hierarchy preservation in OCR
- ✅ Rate limiting with delays
- ✅ Comprehensive error handling & debugging

**Documentation References:**
- LangChain Google: https://context7.com/langchain-ai/langchain-google/
- Google GenAI SDK: https://github.com/googleapis/python-genai

In [8]:
import fitz  # PyMuPDF
import os
import base64
from PIL import Image
import io
from dotenv import load_dotenv
import time
import json

# LangChain imports
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables
load_dotenv()

# Get API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please set it in your .env file.")

print("✓ All imports successful!")
print(f"✓ API Key loaded: {GOOGLE_API_KEY[:10]}...")

✓ All imports successful!
✓ API Key loaded: AIzaSyBEDl...


## Helper Functions

In [9]:
def image_to_base64(image: Image.Image, format="JPEG") -> str:
    """
    Converts a PIL Image to base64 string.
    Handles different image modes (RGBA, Palette) by converting to RGB.
    """
    # Handle images with alpha channels
    if image.mode == 'RGBA':
        bg = Image.new('RGB', image.size, (255, 255, 255))
        bg.paste(image, (0, 0), image)
        image = bg
    elif image.mode == 'P':  # Palette mode
        image = image.convert('RGB')

    buffered = io.BytesIO()
    image.save(buffered, format=format)
    img_bytes = buffered.getvalue()
    return base64.b64encode(img_bytes).decode('utf-8')


def extract_text_from_response(response) -> str:
    """
    Extracts text from Gemini API response.
    Handles cases where response.content is a string or list.
    
    Based on latest LangChain documentation:
    - response.content can be str for text responses
    - response.content can be list when using multimodal outputs
    """
    if not hasattr(response, 'content'):
        return "[ERROR: No 'content' attribute in response]"
    
    content = response.content
    
    # If content is already a string, return it
    if isinstance(content, str):
        return content
    
    # If content is a list, join all text items
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict):
                # Handle dict items (like {'text': '...'})
                if 'text' in item:
                    text_parts.append(item['text'])
        return ''.join(text_parts) if text_parts else "[Empty response]"
    
    # Fallback: convert to string
    return str(content)

## OCR Function with Proper Response Handling

In [10]:
def get_ocr_text_from_image(image_base64: str, api_key: str, debug=False) -> str:
    """
    Sends image to Gemini model for OCR with proper response handling.
    
    Args:
        image_base64: Base64 encoded image string
        api_key: Google API key
        debug: If True, print debug information
        
    Returns:
        Extracted text or error message
    """
    try:
        # Initialize LLM with gemini-2.5-flash (recommended model)
        llm = ChatGoogleGenerativeAI(
            model="gemini-3-flash-preview",
            google_api_key=api_key,
            temperature=0.3  # Lower temperature for consistent OCR
        )
        
        # Create multimodal message with text + image
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "Perform detailed OCR on this image while preserving document structure. "
                        "1. Identify text hierarchy (titles, subtitles, body text) "
                        "2. Preserve formatting (lists, bullet points, numbering) "
                        "3. Maintain text colors and special formatting "
                        "4. Format output in markdown with appropriate heading levels "
                        "5. Extract ALL text exactly as written "
                        "If no text found, respond with: [No text found in image]"
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{image_base64}",
                },
            ]
        )
        
        if debug:
            print("[DEBUG] Sending request to Gemini API...")
        
        # Add rate limiting delay
        time.sleep(1.5)
        
        # Invoke model and get response
        response = llm.invoke([message])
        
        if debug:
            print(f"[DEBUG] Response type: {type(response)}")
            print(f"[DEBUG] Response.content type: {type(response.content)}")
        
        # Extract text using robust handler
        text = extract_text_from_response(response)
        
        # Check if extraction failed
        if "[no text found in image]" in text.lower():
            return "[No text found in image]"
        
        return text
        
    except Exception as e:
        error_msg = f"[OCR Failed: {str(e)}]"
        if debug:
            print(f"[ERROR] {error_msg}")
            import traceback
            traceback.print_exc()
        return error_msg

## PDF Processing Functions

In [11]:
def process_pdf_page_with_ocr(page: fitz.Page, llm_api_key: str, debug=False) -> list:
    """
    Extract images from PDF page and perform OCR on each.
    """
    ocr_results = []
    
    # Get all images on page
    image_list = page.get_images(full=True)
    page_num = page.number + 1
    
    if debug:
        print(f"[PAGE {page_num}] Found {len(image_list)} images")
    
    for img_index, img_info in enumerate(image_list, 1):
        xref = img_info[0]
        
        try:
            # Extract image from PDF
            base_image = page.parent.extract_image(xref)
            if not base_image:
                if debug:
                    print(f"  Image {img_index}: Extraction failed")
                continue
            
            image_bytes = base_image.get("image")
            if not image_bytes:
                if debug:
                    print(f"  Image {img_index}: No image data")
                continue
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes))
            
            # Convert to base64
            img_base64 = image_to_base64(pil_image, format="JPEG")
            
            if debug:
                print(f"  Image {img_index}: Sending to OCR...")
            
            # Perform OCR
            ocr_text = get_ocr_text_from_image(img_base64, llm_api_key, debug=debug)
            
            # Store result if valid
            if ocr_text.strip() and "[no text found" not in ocr_text.lower():
                ocr_results.append(ocr_text.strip())
                if debug:
                    print(f"  Image {img_index}: OCR successful ({len(ocr_text)} chars)")
            else:
                if debug:
                    print(f"  Image {img_index}: No text extracted")
                    
        except Exception as e:
            if debug:
                print(f"  Image {img_index}: Error - {e}")
    
    return ocr_results

## Testing Single Page

In [12]:
# Configuration for testing
PDF_PATH = "CourseBook_Semester3_AlTafsir.pdf"  # Change to your PDF
TEST_PAGE = 1  # Test on page 1

# Test if PDF exists
if os.path.exists(PDF_PATH):
    print(f"✓ PDF found: {PDF_PATH}")
else:
    print(f"✗ PDF not found: {PDF_PATH}")
    print(f"  Available files: {os.listdir('.')}")

# Get PDF info
if os.path.exists(PDF_PATH):
    doc = fitz.open(PDF_PATH)
    print(f"✓ PDF has {len(doc)} pages")
    doc.close()

✓ PDF found: CourseBook_Semester3_AlTafsir.pdf
✓ PDF has 102 pages


In [13]:
# Test single page OCR with debugging
print(f"\n{'='*60}")
print(f"Testing OCR on Page {TEST_PAGE}")
print(f"{'='*60}\n")

try:
    if not os.path.exists(PDF_PATH):
        print(f"ERROR: PDF not found at {PDF_PATH}")
    else:
        doc = fitz.open(PDF_PATH)
        
        # Validate page number
        if TEST_PAGE < 1 or TEST_PAGE > len(doc):
            print(f"ERROR: Page {TEST_PAGE} out of range (1-{len(doc)})")
        else:
            # Load page (convert to 0-based)
            page = doc.load_page(TEST_PAGE - 1)
            
            # Process with OCR (debug=True for verbose output)
            results = process_pdf_page_with_ocr(page, GOOGLE_API_KEY, debug=True)
            
            print(f"\n{'='*60}")
            print(f"Results: {len(results)} text block(s) extracted")
            print(f"{'='*60}\n")
            
            if results:
                for idx, text in enumerate(results, 1):
                    print(f"\n--- Text Block {idx} (First 300 chars) ---")
                    print(text[:300])
                    if len(text) > 300:
                        print(f"... [Total: {len(text)} characters]")
            else:
                print("No text extracted from page")
        
        doc.close()
        
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()


Testing OCR on Page 1

[PAGE 1] Found 1 images
  Image 1: Sending to OCR...
[DEBUG] Sending request to Gemini API...
[DEBUG] Response type: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] Response.content type: <class 'list'>
  Image 1: OCR successful (867 chars)

Results: 1 text block(s) extracted


--- Text Block 1 (First 300 chars) ---
# Tafsir
## Quranic Exegesis

---

### 3 LEVEL

---

![Book Covers and Manuscripts]

#### Left Book Cover:
*   **رسالة دكتوراه** (Doctoral Thesis)
*   **تفسير الإمام الشافعي** (Tafsir al-Imam al-Shafi'i)
*   **لأبي عبد الله محمد بن إدريس المطلبي القرشي** (By Abu Abdullah Muhammad bin Idris al-Muttal
... [Total: 867 characters]


## Test Different Pages

In [14]:
# Test multiple pages
TEST_PAGES = [1, 2, 3]  # Change this to test different pages

if os.path.exists(PDF_PATH):
    doc = fitz.open(PDF_PATH)
    
    for page_num in TEST_PAGES:
        if page_num < 1 or page_num > len(doc):
            print(f"Skipping page {page_num} (out of range)")
            continue
        
        print(f"\n{'='*60}")
        print(f"Testing Page {page_num}")
        print(f"{'='*60}")
        
        page = doc.load_page(page_num - 1)
        results = process_pdf_page_with_ocr(page, GOOGLE_API_KEY, debug=False)
        
        print(f"✓ Extracted {len(results)} text block(s)")
        for idx, text in enumerate(results, 1):
            print(f"  Block {idx}: {len(text)} characters")
    
    doc.close()
else:
    print(f"PDF not found: {PDF_PATH}")


Testing Page 1
✓ Extracted 1 text block(s)
  Block 1: 1088 characters

Testing Page 2
✓ Extracted 1 text block(s)
  Block 1: 111 characters

Testing Page 3
✓ Extracted 1 text block(s)
  Block 1: 16 characters
